# 11 - Churn Model Training (MLflow)
Trains and compares 2 models on the 2017-12-01 snapshot, logs everything to MLflow,
and registers the best one in Unity Catalog as `retail_lakehouse.ml.churn_model@champion`.

**Class imbalance:** ~97%+ of customers churn, so accuracy is meaningless.
Models use class weighting and are judged on:
- **ROC-AUC:** overall ranking quality
- **PR-AUC (returning customers):** how well we find the rare customers who come back
- **Lift @ top 10%:** among the 10% least likely to churn, how many times more returners than average

In [0]:
import importlib
for pkg in ["sklearn", "mlflow", "pandas", "numpy"]:
    try:
        m = importlib.import_module(pkg)
        print(f"✅ {pkg:8} {m.__version__}")
    except ImportError:
        print(f"❌ {pkg:8} MISSING")

In [0]:
import numpy as np
import pandas as pd

C = "retail_lakehouse"
TRAIN_SNAPSHOT = "2017-12-01"

NUMERIC = [
    "recency_days", "tenure_days", "frequency", "monetary", "avg_order_value",
    "avg_items_per_order", "freight_share", "avg_delivery_days", "late_delivery_rate",
    "avg_installments", "credit_card_share", "voucher_share", "distinct_categories",
]
CATEGORICAL = ["customer_state"]
FEATURES = NUMERIC + CATEGORICAL

df = spark.sql(f"""
  SELECT f.*, l.churned
  FROM {C}.ml.customer_features f
  JOIN {C}.ml.churn_labels l
    ON f.customer_key = l.customer_key AND f.snapshot_date = l.snapshot_date
  WHERE f.snapshot_date = DATE'{TRAIN_SNAPSHOT}'
""").toPandas()

X = df[FEATURES]
y = df["churned"].astype(int)

print(f"Training rows: {len(df):,}")
print(f"Churn rate:    {y.mean():.2%}   (returners: {(1 - y).sum():,})")

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), NUMERIC),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL),
    ])

CANDIDATES = {
    "logistic_regression": LogisticRegression(class_weight="balanced", max_iter=2000),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=20,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1,
    ),
}

def evaluate(y_true, churn_prob):
    """Metrics focused on the RARE class: customers who return."""
    returned = 1 - y_true.values
    return_score = 1 - churn_prob
    base_rate = returned.mean()
    top_n = max(1, int(len(returned) * 0.10))
    top_idx = np.argsort(-return_score)[:top_n]          # 10% most likely to return
    return {
        "roc_auc": roc_auc_score(y_true, churn_prob),
        "pr_auc_returning": average_precision_score(returned, return_score),
        "baseline_return_rate": base_rate,
        "lift_top_10pct": returned[top_idx].mean() / base_rate if base_rate > 0 else 0.0,
    }

In [0]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/retail_churn_experiment")

results = {}
for name, estimator in CANDIDATES.items():
    with mlflow.start_run(run_name=name) as run:
        model = Pipeline([("prep", make_preprocessor()), ("clf", estimator)])
        model.fit(X_train, y_train)

        churn_prob = model.predict_proba(X_test)[:, 1]
        metrics = evaluate(y_test, churn_prob)

        mlflow.log_params({
            "model_type": name,
            "train_snapshot": TRAIN_SNAPSHOT,
            "label_window_days": 270,
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": len(FEATURES),
            **{f"clf__{k}": v for k, v in estimator.get_params().items()
               if k in ("class_weight", "n_estimators", "max_depth", "min_samples_leaf", "max_iter")},
        })
        mlflow.log_metrics(metrics)

        # Feature importance (random forest) or coefficient size (logistic regression)
        names = model.named_steps["prep"].get_feature_names_out()
        clf = model.named_steps["clf"]
        weights = clf.feature_importances_ if hasattr(clf, "feature_importances_") else np.abs(clf.coef_[0])
        fi = (pd.DataFrame({"feature": names, "importance": weights})
                .sort_values("importance", ascending=False))
        mlflow.log_text(fi.to_csv(index=False), "feature_importance.csv")

        signature = infer_signature(X_train.head(100), model.predict(X_train.head(100)))
        mlflow.sklearn.log_model(model, "model", signature=signature, input_example=X_train.head(5))

        results[name] = {"run_id": run.info.run_id, **metrics}
        print(f"✅ {name:20} ROC-AUC={metrics['roc_auc']:.3f}  "
              f"PR-AUC(returning)={metrics['pr_auc_returning']:.4f}  "
              f"lift@10%={metrics['lift_top_10pct']:.2f}x  "
              f"(baseline return rate {metrics['baseline_return_rate']:.2%})")

comparison = pd.DataFrame(results).T
display(comparison.reset_index().rename(columns={"index": "model"}))

In [0]:
from mlflow import MlflowClient

MODEL_NAME = f"{C}.ml.churn_model"
best_name = comparison["pr_auc_returning"].astype(float).idxmax()
best_run_id = results[best_name]["run_id"]

mv = mlflow.register_model(f"runs:/{best_run_id}/model", MODEL_NAME)

client = MlflowClient()
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
client.update_registered_model(
    MODEL_NAME,
    description="Predicts probability a customer will NOT purchase again within 270 days. "
                "Trained on point-in-time features (retail_lakehouse.ml.customer_features). "
                "Champion selected by PR-AUC on returning customers.",
)
client.update_model_version(
    MODEL_NAME, mv.version,
    description=f"{best_name} | ROC-AUC {results[best_name]['roc_auc']:.3f} | "
                f"PR-AUC(returning) {results[best_name]['pr_auc_returning']:.4f} | "
                f"lift@10% {results[best_name]['lift_top_10pct']:.2f}x",
)

print(f"🏆 Champion: {best_name} -> {MODEL_NAME} version {mv.version} (alias @champion)")